# AI 圖片分類專案：使用 CNN 分類 CIFAR-10 圖片

**執行平台：** Google Colab  
**主要工具：** Python、TensorFlow / Keras、Matplotlib  
**任務：** 將圖片分類為 10 種類別之一。

本專案採用 CIFAR-10 公開資料集。資料集包含 10 個類別的彩色小圖片。本筆記本會依序載入資料、檢視圖片、建立 CNN 模型、訓練模型、評估準確率，並展示預測結果。

> 建議在 Colab 選擇 GPU 執行階段；若沒有 GPU，也可以使用 CPU 執行，只是訓練時間可能較長。

## 1. 設定執行環境

在 Google Colab 上方選單選擇「執行階段 → 變更執行階段類型」，硬體加速器可選 GPU（若帳戶目前提供）。

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow 版本：", tf.__version__)
print("可用 GPU：", tf.config.list_physical_devices("GPU"))

## 2. 載入 CIFAR-10 資料集

CIFAR-10 有 10 個類別：飛機、汽車、鳥、貓、鹿、狗、青蛙、馬、船、卡車。資料集會由 Keras 自動下載。

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("訓練圖片數量：", len(x_train))
print("測試圖片數量：", len(x_test))
print("圖片尺寸：", x_train.shape[1:])

## 3. 資料預處理與圖片檢視

將像素值從 0～255 正規化到 0～1，讓模型較容易學習。

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i][0])])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. 建立 CNN 圖片分類模型

CNN（卷積神經網路）常用於圖片辨識。此範例使用卷積層、池化層、Dropout 與全連接層建立分類模型。

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. 訓練模型

先設定 5 個訓練週期（epochs）作為示範。若想觀察更充分的學習效果，可以增加 epochs，但訓練時間也會增加。

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1
)

## 6. 繪製訓練過程

觀察訓練集與驗證集的準確率、損失值變化。

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 7. 評估模型

使用測試資料評估模型表現。實際準確率會依執行環境、訓練次數及模型狀況而有所不同。

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"測試集 Loss：{test_loss:.4f}")
print(f"測試集 Accuracy：{test_accuracy:.4f}")
print(f"測試集準確率：{test_accuracy * 100:.2f}%")

## 8. 顯示圖片分類預測結果

顯示部分測試圖片、模型預測類別與正確類別，方便比較分類結果。

In [ ]:
predictions = model.predict(x_test[:12], verbose=0)

plt.figure(figsize=(12, 8))
for i in range(12):
    predicted_label = int(np.argmax(predictions[i]))
    true_label = int(y_test[i][0])
    color = "green" if predicted_label == true_label else "red"

    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])
    plt.title(
        f"Pred: {class_names[predicted_label]}\nTrue: {class_names[true_label]}",
        color=color
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 9. 儲存模型

將訓練完成的模型儲存為 Keras 格式。Colab 工作階段中的檔案可由左側「檔案」面板下載；若要長期保存，可另行掛載 Google Drive。

In [ ]:
model.save("cifar10_image_classifier.keras")
print("模型已儲存：cifar10_image_classifier.keras")

## 10. 專案結論

本專案使用 Google Colab 執行 TensorFlow 圖片分類流程，完成資料集載入、資料正規化、CNN 模型建立、模型訓練、準確率評估及預測結果展示。

**注意：** 本範例屬於教學示範，模型準確率不代表可直接用於實際產品。若要提升效果，可以增加訓練週期、進行資料增強、調整模型結構，並比較不同模型的結果。

## 參考資料
- CIFAR-10： https://www.cs.toronto.edu/~kriz/cifar.html
- TensorFlow 教學： https://www.tensorflow.org/tutorials/images/cnn
- Google Colab： https://colab.research.google.com/
- Colab 與 GitHub 操作參考（課程指定文章）： https://blog.csdn.net/qq_51070956/article/details/134503497